In [5]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report

In [4]:
# Correct paths
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Models folder:", MODEL_DIR)

Data folder: ..\data
Models folder: ..\models


In [6]:
np.random.seed(42)

n = 2000

loan_data = pd.DataFrame({
    "age": np.random.randint(21, 60, n),
    "monthly_income": np.random.randint(15000, 200000, n),
    "loan_amount": np.random.randint(50000, 2000000, n),
    "credit_history_years": np.random.randint(0, 15, n),
    "existing_debt": np.random.randint(0, 800000, n),
    "employment_years": np.random.randint(0, 20, n)
})

loan_data["debt_to_income"] = loan_data["existing_debt"] / loan_data["monthly_income"]

loan_data["loan_approved"] = (
    (loan_data["monthly_income"] > 35000) &
    (loan_data["credit_history_years"] >= 2) &
    (loan_data["debt_to_income"] < 12) &
    (loan_data["loan_amount"] < loan_data["monthly_income"] * 35)
).astype(int)

loan_data.to_csv(os.path.join(DATA_DIR, "loan_data.csv"), index=False)

loan_data.head()

,age,monthly_income,loan_amount,credit_history_years,existing_debt,employment_years,debt_to_income,loan_approved
0,59,63231,1448949,12,88741,12,1.403441,1
1,49,44301,1233765,10,18830,1,0.425047,1
2,35,187395,1713129,10,317188,6,1.692617,1
3,28,33752,572154,4,69202,14,2.050308,0
4,41,113452,672006,3,299189,10,2.637142,1


In [7]:
X = loan_data[
    [
        "age",
        "monthly_income",
        "loan_amount",
        "credit_history_years",
        "existing_debt",
        "employment_years"
    ]
]

y = loan_data["loan_approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

loan_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

loan_model.fit(X_train, y_train)

loan_pred = loan_model.predict(X_test)

print("Loan Model Accuracy:", accuracy_score(y_test, loan_pred))
print(classification_report(y_test, loan_pred))

joblib.dump(loan_model, os.path.join(MODEL_DIR, "loan_model.pkl"))

print("loan_model.pkl saved successfully")

Loan Model Accuracy: 0.9875
              precision    recall  f1-score   support

           0       0.98      0.97      0.98       112
           1       0.99      0.99      0.99       288

    accuracy                           0.99       400
   macro avg       0.99      0.98      0.98       400
weighted avg       0.99      0.99      0.99       400

loan_model.pkl saved successfully


In [8]:
n = 3000

fraud_data = pd.DataFrame({
    "transaction_amount": np.random.randint(100, 200000, n),
    "transaction_hour": np.random.randint(0, 24, n),
    "location_risk": np.random.randint(1, 10, n),
    "device_risk": np.random.randint(1, 10, n),
    "previous_failed_attempts": np.random.randint(0, 8, n)
})

fraud_data["is_fraud"] = (
    (fraud_data["transaction_amount"] > 90000) |
    ((fraud_data["transaction_hour"] >= 0) & (fraud_data["transaction_hour"] <= 4) & (fraud_data["location_risk"] > 6)) |
    ((fraud_data["device_risk"] > 7) & (fraud_data["previous_failed_attempts"] > 3))
).astype(int)

fraud_data.to_csv(os.path.join(DATA_DIR, "fraud_data.csv"), index=False)

fraud_data.head()

,transaction_amount,transaction_hour,location_risk,device_risk,previous_failed_attempts,is_fraud
0,62557,21,3,2,5,0
1,166515,5,2,8,4,1
2,193561,13,8,7,1,1
3,139828,3,6,5,0,1
4,5841,21,7,2,2,0


In [9]:
X = fraud_data[
    [
        "transaction_amount",
        "transaction_hour",
        "location_risk",
        "device_risk",
        "previous_failed_attempts"
    ]
]

y = fraud_data["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

fraud_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

fraud_model.fit(X_train, y_train)

fraud_pred = fraud_model.predict(X_test)

print("Fraud Model Accuracy:", accuracy_score(y_test, fraud_pred))
print(classification_report(y_test, fraud_pred))

joblib.dump(fraud_model, os.path.join(MODEL_DIR, "fraud_model.pkl"))

print("fraud_model.pkl saved successfully")

Fraud Model Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       226
           1       1.00      1.00      1.00       374

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600

fraud_model.pkl saved successfully


In [10]:
n = 2500

credit_data = pd.DataFrame({
    "monthly_income": np.random.randint(15000, 250000, n),
    "repayment_score": np.random.randint(1, 100, n),
    "existing_debt": np.random.randint(0, 1000000, n),
    "credit_history_years": np.random.randint(0, 20, n),
    "late_payments": np.random.randint(0, 12, n)
})

credit_data["credit_score"] = (
    300
    + credit_data["repayment_score"] * 3
    + credit_data["credit_history_years"] * 10
    + credit_data["monthly_income"] / 2000
    - credit_data["late_payments"] * 15
    - credit_data["existing_debt"] / 25000
)

credit_data["credit_score"] = credit_data["credit_score"].clip(300, 900).astype(int)

credit_data.to_csv(os.path.join(DATA_DIR, "credit_score_data.csv"), index=False)

credit_data.head()

,monthly_income,repayment_score,existing_debt,credit_history_years,late_payments,credit_score
0,165111,51,123793,19,11,555
1,29017,76,370059,17,0,697
2,97901,37,15096,19,1,634
3,128000,31,846479,18,8,483
4,54215,62,735002,18,7,558


In [11]:
X = credit_data[
    [
        "monthly_income",
        "repayment_score",
        "existing_debt",
        "credit_history_years",
        "late_payments"
    ]
]

y = credit_data["credit_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

credit_model = RandomForestRegressor(
    n_estimators=150,
    random_state=42
)

credit_model.fit(X_train, y_train)

joblib.dump(credit_model, os.path.join(MODEL_DIR, "credit_model.pkl"))

print("credit_model.pkl saved successfully")

credit_model.pkl saved successfully
